# 25882 AI-powered Investment and Risk Management — Assessment 1: Empirical Assignment

**Group members (name, student ID):**
- TODO: Name 1, Student ID 1
- TODO: Name 2, Student ID 2 *(delete this line if working alone)*


## Part B choices

TODO — once selected, state explicitly which two extensions we are attempting, e.g.:

> We attempt **B1 (Portfolio optimisation)** from Category B and **A2 (Factor analysis)** from Category A,
> satisfying the requirement that the two options come from different categories and at least one comes
> from Category A or B.


In [ ]:
import importlib.util
import subprocess
import sys


def _ensure_installed(packages):
    """
    Cheap safety net for `Kernel -> Restart & Run All` on a machine where the
    notebook's kernel does not match the environment `pip install -r
    requirements.txt` was run into (a common Jupyter/conda mix-up). Checks
    each package's presence in *this* kernel only -- not its version -- and
    installs only what is actually missing, into `sys.executable` so it
    cannot disagree with itself. A no-op, and silent, when everything is
    already available -- the expected case in a correctly set-up environment.
    """
    missing = [p for p in packages if importlib.util.find_spec(p) is None]
    if missing:
        print(f"Installing missing packages into this kernel: {missing}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


_ensure_installed(["numpy", "pandas", "yfinance"])

import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR = Path("data_cache")
DATA_DIR.mkdir(exist_ok=True)

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
print(f"Notebook environment set up. Random seed fixed at {RANDOM_SEED}.")


## A1 — Universe selection and data acquisition

### Asset universe and justification

| Ticker | Name | Sector | Exchange | Currency |
|---|---|---|---|---|
| AAPL | Apple Inc. | Information Technology | NASDAQ | USD |
| JPM | JPMorgan Chase & Co. | Financials | NYSE | USD |
| XOM | ExxonMobil Corp. | Energy | NYSE | USD |
| JNJ | Johnson & Johnson | Health Care | NYSE | USD |
| 7203.T | Toyota Motor Corp. | Consumer Discretionary (Automobiles) | Tokyo Stock Exchange | JPY |

We chose four large-cap US stocks spanning four distinct GICS sectors (technology, financials, energy,
healthcare) plus one Japan-listed stock (Toyota, 7203.T) to satisfy two goals at once. First, four
distinct sectors avoid the degenerate, near-perfectly-correlated universe a single-sector selection
would produce, which the brief warns would flatten the diversification and optimisation results in
Part B. Second, Toyota is listed on the Tokyo Stock Exchange and trades in JPY, giving us a genuine
cross-currency, cross-trading-calendar asset — required by the brief, and necessary to make the
currency-alignment and holiday-calendar corrections in Section A2 non-vacuous. A different choice —
five US large-caps in USD, say — would have prevented us from ever observing an FX or calendar-alignment
effect at all, and would have understated how much diversification the equal-weight benchmark in A4
can actually deliver.


In [ ]:
# --- Universe and sample period ---
TICKERS = ["AAPL", "JPM", "XOM", "JNJ", "7203.T"]

# Fixed, hard-coded date range (not "today") so the notebook is exactly
# reproducible on re-run: the price data requested does not depend on when
# the notebook happens to be executed. This spans just over 9 years, safely
# above the 8-year minimum.
PRICE_START = "2016-01-01"
PRICE_END = "2025-01-01"

# Recorded once, manually, at the time of the first successful live download
# below -- this is metadata about *when we pulled the data*, not a parameter
# that should silently change what gets downloaded on a later re-run.
DOWNLOAD_DATE_RECORDED = "2026-09-15"  # TODO: update to the actual date you first run this successfully

print(f"Universe: {TICKERS}")
print(f"Requested date range: {PRICE_START} to {PRICE_END}")


In [ ]:
def load_price_panel(tickers, start, end, cache_dir=DATA_DIR, force_download=False):
    """
    Download daily unadjusted and dividend/split-adjusted close prices for
    `tickers` between `start` and `end`, and return them as two clean, wide,
    date-aligned DataFrames (one column per ticker).

    Used by every later section of this notebook -- built once here, reused
    throughout (including for the FX and risk-free single-ticker series in
    A2), per the assignment's instruction not to re-implement this.

    Behaviour
    ---------
    * yfinance now defaults to auto_adjust=True, which returns only an
      adjusted Close and discards the raw price. We call it with
      auto_adjust=False explicitly so both series are available, because
      Section A2 needs both.
    * yfinance returns MultiIndex (ticker, field) columns for a multi-ticker
      request, but sometimes returns FLAT columns for a single-ticker
      request even with group_by="ticker" -- both shapes are handled.
    * A ticker that returns nothing, or returns an all-NaN Close column, is
      dropped and reported -- *before* any row-wise NaN handling. Dropping
      NaN rows first would let one broken ticker delete the entire sample.
    * On success, the two wide panels are cached to `cache_dir` as CSV files.
      On a later call (or a later notebook re-run) with the cache already
      present, the function reads the cache directly and does not touch the
      network at all.
    * If a live download raises (network error, rate limit, vendor outage --
      "which happens", per the brief) the function falls back to the local
      cache if one exists, so the notebook degrades gracefully rather than
      crashing on a clean re-run at marking time.

    Parameters
    ----------
    tickers : list of str
    start, end : str, "YYYY-MM-DD"
    cache_dir : Path
    force_download : bool
        Skip a fresh-cache check and always attempt a live download first
        (still falls back to the cache on failure).

    Returns
    -------
    raw_close : DataFrame, date-indexed, one column per surviving ticker
    adj_close : DataFrame, date-indexed, one column per surviving ticker
    log : dict
        Provenance metadata: source, timestamp, tickers requested vs
        returned, yfinance version.
    """
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    raw_path = cache_dir / "raw_close.csv"
    adj_path = cache_dir / "adj_close.csv"

    if not force_download and raw_path.exists() and adj_path.exists():
        raw_close = pd.read_csv(raw_path, index_col=0, parse_dates=True)
        adj_close = pd.read_csv(adj_path, index_col=0, parse_dates=True)
        log = {
            "source": "local cache",
            "path": str(cache_dir),
            "tickers": list(raw_close.columns),
        }
        print(f"Loaded cached panel from {cache_dir}/ "
              f"(delete raw_close.csv / adj_close.csv there to force a fresh download).")
        return raw_close, adj_close, log

    try:
        data = yf.download(tickers, start=start, end=end, auto_adjust=False,
                            group_by="ticker", progress=False, threads=True)
        if data.empty:
            raise ValueError("yfinance returned an empty frame for every ticker")

        raw_cols, adj_cols = {}, {}
        if isinstance(data.columns, pd.MultiIndex):
            for t in tickers:
                try:
                    sub = data[t]
                except KeyError:
                    print(f"  WARNING: no data at all returned for {t}; dropping from panel")
                    continue
                if sub["Close"].dropna().empty:
                    print(f"  WARNING: {t} returned an all-NaN Close column; dropping from panel")
                    continue
                raw_cols[t] = sub["Close"]
                adj_cols[t] = sub["Adj Close"]
        else:
            # Flat-column response: only valid for a single requested ticker.
            if len(tickers) != 1:
                raise ValueError(
                    f"Expected MultiIndex columns for {len(tickers)} tickers, got flat "
                    f"columns {list(data.columns)}"
                )
            t = tickers[0]
            if data["Close"].dropna().empty:
                print(f"  WARNING: {t} returned an all-NaN Close column; dropping from panel")
            else:
                raw_cols[t] = data["Close"]
                adj_cols[t] = data["Adj Close"]

        if not raw_cols:
            raise ValueError("Every requested ticker came back empty or all-NaN")

        raw_close = pd.DataFrame(raw_cols).sort_index()
        adj_close = pd.DataFrame(adj_cols).sort_index()

        raw_close.to_csv(raw_path)
        adj_close.to_csv(adj_path)

        log = {
            "source": "yfinance (live download)",
            "download_timestamp": datetime.now().isoformat(timespec="seconds"),
            "yfinance_version": getattr(yf, "__version__", "unknown"),
            "requested_tickers": list(tickers),
            "returned_tickers": list(raw_close.columns),
            "requested_start": start,
            "requested_end": end,
        }
        print(f"Downloaded fresh data and cached it to {cache_dir}/.")
        return raw_close, adj_close, log

    except Exception as exc:
        print(f"Live download failed ({exc!r}).")
        if raw_path.exists() and adj_path.exists():
            print("Falling back to the previously cached local files so the "
                  "notebook can still run end to end.")
            raw_close = pd.read_csv(raw_path, index_col=0, parse_dates=True)
            adj_close = pd.read_csv(adj_path, index_col=0, parse_dates=True)
            log = {
                "source": "local cache (after a failed live download)",
                "path": str(cache_dir),
            }
            return raw_close, adj_close, log
        raise RuntimeError(
            f"No live data available and no local cache found at {cache_dir}/. "
            "Cannot proceed -- see the note on reproducibility in the assignment brief."
        ) from exc


In [ ]:
raw_close, adj_close, download_log = load_price_panel(TICKERS, PRICE_START, PRICE_END)

print("\nDownload log:")
for k, v in download_log.items():
    print(f"  {k}: {v}")


In [ ]:
def summarize_panel(price_df, label="", min_years=8):
    """
    Report, per asset: first date, last date, observation count, and years
    spanned -- and flag any asset whose history falls short of `min_years`.

    This is the "inspect what arrived before using it" step: it does not
    silently trust the download, it shows the evidence.
    """
    summary = pd.DataFrame({
        "first_date": price_df.apply(lambda s: s.dropna().index.min()),
        "last_date": price_df.apply(lambda s: s.dropna().index.max()),
        "n_obs": price_df.count(),
    })
    summary["years_span"] = (summary["last_date"] - summary["first_date"]).dt.days / 365.25

    print(f"--- {label} ---")
    display(summary)

    short = summary[summary["years_span"] < min_years]
    if not short.empty:
        print(f"WARNING: history shorter than {min_years} years for: {list(short.index)}")
    else:
        print(f"All assets meet the {min_years}-year minimum.")
    return summary


raw_summary = summarize_panel(raw_close, label="Raw close -- inspection", min_years=8)
adj_summary = summarize_panel(adj_close, label="Adjusted close -- inspection", min_years=8)


In [ ]:
ASSET_METADATA = {
    "AAPL":   {"name": "Apple Inc.",            "sector": "Information Technology",
               "exchange": "NASDAQ", "currency": "USD"},
    "JPM":    {"name": "JPMorgan Chase & Co.",  "sector": "Financials",
               "exchange": "NYSE",   "currency": "USD"},
    "XOM":    {"name": "ExxonMobil Corp.",      "sector": "Energy",
               "exchange": "NYSE",   "currency": "USD"},
    "JNJ":    {"name": "Johnson & Johnson",     "sector": "Health Care",
               "exchange": "NYSE",   "currency": "USD"},
    "7203.T": {"name": "Toyota Motor Corp.",    "sector": "Consumer Discretionary (Automobiles)",
               "exchange": "Tokyo Stock Exchange", "currency": "JPY"},
}

asset_summary = (
    pd.DataFrame(ASSET_METADATA).T
    .join(raw_summary[["first_date", "last_date", "n_obs", "years_span"]])
)
asset_summary.index.name = "ticker"
asset_summary["years_span"] = asset_summary["years_span"].round(3)

display(asset_summary)

n_obs_gap = asset_summary["n_obs"].max() - asset_summary["n_obs"].min()
if n_obs_gap > 0:
    shortest = asset_summary["n_obs"].idxmin()
    print(f"\n{shortest} has {n_obs_gap} fewer trading-day observations than the fullest "
          f"series over the same nominal date range -- direct evidence that its exchange "
          f"does not share a trading calendar with the US tickers.")


In [ ]:
def load_ohlcv_preview(tickers, start, end, cache_dir=DATA_DIR / "ohlcv_preview",
                       force_download=False):
    """
    Download the FULL OHLCV panel (Open, High, Low, Close, Adj Close, Volume)
    purely for visual inspection of what the vendor actually served.

    This is display-only and is NOT what any later section computes from --
    every later section works from load_price_panel()'s raw_close/adj_close,
    which deliberately keeps only Close and Adj Close (A2 is specifically
    about auditing those two series). Same cache-then-live-then-graceful-
    skip discipline as the main loader, but a failure here does not raise:
    it is a display convenience, not something the rest of the notebook
    depends on.
    """
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / "ohlcv_long.csv"

    if not force_download and cache_path.exists():
        return pd.read_csv(cache_path, parse_dates=["date"])

    try:
        data = yf.download(tickers, start=start, end=end, auto_adjust=False,
                            group_by="ticker", progress=False, threads=True)
        if data.empty:
            raise ValueError("yfinance returned an empty frame")

        frames = []
        if isinstance(data.columns, pd.MultiIndex):
            for t in tickers:
                if t not in data.columns.get_level_values(0):
                    continue
                sub = data[t].reset_index().rename(columns={"Date": "date"})
                sub["ticker"] = t
                frames.append(sub)
        else:
            sub = data.reset_index().rename(columns={"Date": "date"})
            sub["ticker"] = tickers[0]
            frames.append(sub)

        long = pd.concat(frames, ignore_index=True)
        long = long[["date", "ticker", "Open", "High", "Low", "Close", "Adj Close", "Volume"]]
        long.to_csv(cache_path, index=False)
        return long

    except Exception as exc:
        print(f"OHLCV preview download failed ({exc!r}); this cell is display-only, "
              "skipping it gracefully -- it does not affect any other section.")
        return None


ohlcv_long = load_ohlcv_preview(TICKERS, PRICE_START, PRICE_END)

if ohlcv_long is not None:
    print("Sample of the raw downloaded data (first 3 rows per ticker):")
    display(ohlcv_long.groupby("ticker").head(3).set_index(["ticker", "date"]))

    print("\nDescriptive summary per ticker, full sample (Open/High/Low/Close/Adj Close/Volume):")
    ohlcv_stats = (
        ohlcv_long.groupby("ticker")[["Open", "High", "Low", "Close", "Adj Close", "Volume"]]
        .agg(["mean", "min", "max"])
        .round(2)
    )
    display(ohlcv_stats)


### A1 discussion

TODO once run with live data: comment on the actual date ranges and observation counts returned --
in particular, whether Toyota's calendar (Tokyo trading days) gives a different `n_obs` from the
US tickers even over the same nominal date range, and whether any ticker's history fell short of
the 8-year requirement.

**Note on this notebook's execution environment:** the sandbox used to draft this notebook has no
outbound network access to Yahoo Finance, so `load_price_panel()` above could not perform a live
download here -- its logic was verified separately against a mocked `yfinance.download` covering
(1) a normal multi-ticker download, (2) a ticker returning an all-NaN column being dropped before
any row-level NaN handling, (3) a cache-hit skipping the network entirely, and (4) a live-download
failure falling back to a previously cached panel. Run **Kernel → Restart & Run All** on a machine
with internet access to perform the actual download, populate `data_cache/raw_close.csv` and
`data_cache/adj_close.csv`, and replace this note with the real inspection results.


## A2 — Data and convention audit: the correction waterfall

We build a deliberately careless baseline, then apply one correction at a time -- each on
top of everything already corrected before it -- recomputing annualised return, annualised
volatility, Sharpe ratio and maximum drawdown after every step. The portfolio construction
method (equal weight, no explicit rebalancing logic beyond what `pct_change` implies) is
held fixed throughout, so the *only* thing changing row to row is the data/convention
correction under test.


In [ ]:
def equal_weight_portfolio_returns(price_panel, return_type="simple"):
    """
    Equal-weight (1/N), implicitly-daily-rebalanced SIMPLE portfolio return
    -- the weighted average of each asset's own simple return, which is the
    exact quantity for a dollar-weighted rebalanced portfolio.

    return_type="log" does NOT average individual assets' own log returns:
    mean(log(1+r_i)) != log(1 + mean(r_i)) whenever the r_i differ (Jensen's
    inequality on the concave log), so that would silently describe a
    DIFFERENT, not-quite-real portfolio rather than this one on a log scale
    -- confirmed on a synthetic panel with an injected single-asset shock,
    where that mistake alone moved max drawdown from -16.96% to -20.55%
    purely as an averaging artifact, nothing to do with simple-vs-log.
    Instead we take log1p of the portfolio's own simple return: exactly
    this portfolio, expressed on a log scale. The wealth path (and
    therefore drawdown) is then identical to the simple-return version by
    construction; only the annualised-return statistic differs, which is
    the arithmetic-vs-geometric-mean point Step 8 is actually about.

    If a column is NaN on a given date (e.g. a holiday gap we chose not to
    fill), that asset is silently excluded from the mean on that date,
    which reweights the surviving assets to sum to 1 -- itself a
    defensible choice, in contrast to fabricating a price.
    """
    asset_returns = price_panel.pct_change()
    portfolio_simple = asset_returns.mean(axis=1, skipna=True).dropna()
    if return_type == "simple":
        return portfolio_simple
    elif return_type == "log":
        return np.log1p(portfolio_simple)
    else:
        raise ValueError(return_type)


def compute_stats(returns, rf=0.0, periods_per_year=252, return_type="simple"):
    """
    Annualised return, annualised volatility, Sharpe ratio and maximum
    drawdown from a return series.

    Parameters
    ----------
    returns : Series
        Portfolio returns, simple or log per `return_type`.
    rf : float or Series
        Per-period risk-free rate already aligned to `returns`' index. Used
        only for the Sharpe ratio's excess return, not for the headline
        annualised return. Pass 0.0 for the "Sharpe against zero" baseline.
    periods_per_year : float
        252 for the naive assumption; the panel's own observed trading-day
        count for the corrected version (Step 7).
    return_type : "simple" or "log"
        For simple returns the annualised return is the arithmetic mean
        scaled by periods_per_year -- this OVERSTATES the compounded growth
        rate because of volatility drag (Jensen's inequality on the concave
        log-wealth function). For log returns, the arithmetic mean scaled by
        periods_per_year already equals the annualised geometric growth
        rate, which is why switching return_type is this waterfall's
        built-in arithmetic-vs-geometric correction (Step 8).
    """
    returns = returns.dropna()
    if isinstance(rf, pd.Series):
        rf = rf.reindex(returns.index).fillna(0.0)
    excess = returns - rf

    if return_type == "simple":
        wealth = (1 + returns).cumprod()
    elif return_type == "log":
        wealth = np.exp(returns.cumsum())
    else:
        raise ValueError(return_type)

    ann_return = returns.mean() * periods_per_year
    ann_vol = excess.std(ddof=1) * np.sqrt(periods_per_year)
    ann_excess_return = excess.mean() * periods_per_year
    sharpe = ann_excess_return / ann_vol if ann_vol > 0 else np.nan

    running_max = wealth.cummax()
    drawdown = wealth / running_max - 1

    return {
        "ann_return": ann_return,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": drawdown.min(),
    }


waterfall_rows = {}

def record_step(label, portfolio_returns, **kwargs):
    """Compute and stash one waterfall row; returns the stats dict too."""
    stats = compute_stats(portfolio_returns, **kwargs)
    waterfall_rows[label] = stats
    print(label, "->", {k: round(v, 4) for k, v in stats.items()})
    return stats


In [ ]:
def naive_ffill(price_panel):
    """Blindly forward-fill every gap, regardless of why it exists."""
    return price_panel.ffill()


def defensible_fill(price_panel, max_gap=1):
    """
    Forward-fill only short (<= max_gap row) gaps -- the signature of one
    market being closed while another traded (e.g. a Japanese public
    holiday that is not a US holiday, or vice versa), where "price
    unchanged, no trade occurred" is a defensible reading. A longer gap is
    NOT filled and is reported instead of fabricated; A1's loader already
    flagged anything that looked like a broken ticker, so a long gap
    surviving to here is treated as genuinely missing.
    """
    filled = price_panel.ffill(limit=max_gap)
    still_missing = filled.isna().sum()
    if still_missing.sum() > 0:
        print(f"Still missing after a {max_gap}-trading-day defensible fill "
              f"(left as NaN, not fabricated):")
        print(still_missing[still_missing > 0])
    else:
        print(f"No gaps longer than {max_gap} trading day(s) remained after the defensible fill.")
    return filled


In [ ]:
def flag_extreme_returns(asset_returns, z_thresh=6.0):
    """
    Flag single-asset daily returns more than `z_thresh` standard deviations
    from THAT ASSET's own mean -- a per-asset, not a blanket, threshold,
    since assets have very different typical volatility.

    Implemented without DataFrame.stack(), whose default NaN-dropping
    behaviour changed across pandas versions and silently returned every
    cell (including NaNs) rather than only the flagged ones in an earlier
    draft of this function -- verified with a synthetic bad print before
    trusting it on real data.
    """
    z = (asset_returns - asset_returns.mean()) / asset_returns.std(ddof=1)
    mask = z.abs() > z_thresh
    records = []
    for ticker in asset_returns.columns:
        hits = asset_returns.loc[mask[ticker].fillna(False), ticker]
        for date, value in hits.items():
            records.append({"date": date, "ticker": ticker, "return": value,
                             "z_score": z.loc[date, ticker]})
    flagged = pd.DataFrame(records, columns=["date", "ticker", "return", "z_score"])
    return flagged.sort_values("date").reset_index(drop=True)


# Market-wide (or sector-wide) episodes inside our 2016-2025 sample. A
# flagged observation that falls inside one of these windows is presumed a
# genuine move, not a data error, unless investigation of the surrounding
# prices says otherwise.
#
# Revision note: a first pass at this list (2018 sell-off + COVID only) ran
# against our real downloaded data and left 5 flagged observations
# unexplained, which the pipeline then treated as candidate data errors and
# INTERPOLATED OVER -- silently muting three genuine, well-documented events:
# the 9 Nov 2020 Pfizer/BioNTech vaccine-efficacy announcement (a historic
# one-day value/cyclical rally -- JPM and XOM both spiked on it), the 5-6
# Aug 2024 Bank of Japan rate hike and yen carry-trade unwind (the Nikkei's
# worst day since 1987, hitting Toyota directly -- this is the exact episode
# described in this subject's own Lecture 6 slides on procyclicality), and
# the 6 Nov 2024 post-US-election financials rally (JPM, on deregulation
# expectations). None of these is a data error, and none should have been
# smoothed away. The list below is the corrected version; this is exactly
# the sort of silent, plausible-looking mistake the brief's Part C asks you
# to document under "what did not work".
KNOWN_MARKET_EVENTS = [
    ("2018-02-02", "2018-02-09", "Volmageddon -- XIV unwind, VIX spike"),
    ("2018-12-01", "2018-12-26", "Q4 2018 growth-scare sell-off"),
    ("2020-02-20", "2020-04-07", "COVID-19 crash and initial rebound"),
    ("2020-09-01", "2020-09-08", "Sept 2020 tech pullback"),
    ("2020-11-09", "2020-11-09", "Pfizer/BioNTech vaccine efficacy announcement"),
    ("2022-01-01", "2022-10-31", "2022 rate-hike bear market"),
    ("2023-03-08", "2023-03-15", "SVB / regional bank stress"),
    ("2024-08-05", "2024-08-06", "BoJ rate hike / yen carry-trade unwind (see Lecture 6)"),
    ("2024-11-05", "2024-11-08", "2024 US presidential election result"),
]


def classify_flagged(flagged, events=KNOWN_MARKET_EVENTS):
    def event_for(date):
        for start, end, name in events:
            if pd.Timestamp(start) <= date <= pd.Timestamp(end):
                return name
        return None
    flagged = flagged.copy()
    flagged["known_event"] = flagged["date"].apply(event_for)
    return flagged


In [ ]:
def convert_jpy_to_usd(jpy_price_series, usdjpy_rate):
    """
    `usdjpy_rate` (Yahoo ticker "JPY=X") quotes JPY per 1 USD, so dividing a
    JPY price by it gives the equivalent USD price. Rate gaps are
    forward-filled since FX trades continuously and a missing print just
    means "unchanged since the last quote", unlike an equity halt.
    """
    return jpy_price_series / usdjpy_rate.reindex(jpy_price_series.index).ffill()


def align_risk_free(returns_index, rf_series, direction="backward"):
    """
    Align a sparsely-observed risk-free series (T-bill yields are not
    published on every trading day) onto `returns_index` via an as-of merge.

    direction="backward" (correct): use the most recent rate that was
    actually known as of each date -- no future information used.
    direction="forward" (the look-ahead bug): uses the NEXT rate the market
    had not yet seen. Included only to measure the bug's size before
    discarding it.

    Bug fixed here: an earlier version built the "date" column for each
    side differently -- `rename(columns={"index": "date"})` on one side
    assumed `reset_index()` always produces a column literally called
    "index", which is only true when the source index has no name. Our
    price/return series inherit the name "Date" from yfinance, so that
    assumption silently failed on real data (KeyError from merge_asof, not
    caught by the earlier synthetic test because that test used an
    already-unnamed DatetimeIndex). `rename_axis("date")` before
    `reset_index()` sidesteps the issue entirely: the resulting column is
    always called "date", whatever the source index was named or not.
    """
    left = pd.DataFrame(index=returns_index).rename_axis("date").reset_index()
    right = rf_series.sort_index().rename("rf").rename_axis("date").reset_index()
    aligned = pd.merge_asof(left, right, on="date", direction=direction).set_index("date")["rf"]
    return aligned


### Steps 0-1: naive baseline, then dividend/split-adjusted prices

In [ ]:
naive_prices = naive_ffill(raw_close)
r0 = equal_weight_portfolio_returns(naive_prices)
s0 = record_step("0. Naive baseline (raw prices, blind ffill, Sharpe vs 0, 252d, simple/arith.)", r0)

adj_prices_naive_fill = naive_ffill(adj_close)
r1 = equal_weight_portfolio_returns(adj_prices_naive_fill)
s1 = record_step("1. + dividend/split-adjusted prices", r1)


### Step 2: defensible treatment of non-trading days

Toyota (7203.T, Tokyo Stock Exchange) and the four US tickers do not share a holiday
calendar, so the raw joined panel has gaps on US-only and Japan-only holidays alike.
Blindly forward-filling every gap (Step 0/1) treats a several-day data outage the same
as a single foreign holiday. We instead forward-fill only single-day gaps and report
anything longer rather than fabricate it.

In [ ]:
adj_prices_defensible = defensible_fill(adj_close, max_gap=1)
r2 = equal_weight_portfolio_returns(adj_prices_defensible)
s2 = record_step("2. + defensible non-trading-day treatment", r2)


### Step 3: investigate extreme returns

We flag any single-asset daily return more than 6 standard deviations from that asset's
own mean, then check each flagged date against known market-wide events in our sample.
A flag that falls inside a known event window is a genuine market move and is left alone
-- winsorising it would delete real information (per the brief's own warning). A flag
that does **not** correspond to a known event is a candidate data error; we correct those
by interpolating the *price level*, not the return, and only for the specific (date,
ticker) pairs identified.

One diagnostic worth watching for: a genuine one-sided crash does not fully reverse the
next day, while an isolated bad print typically shows as a paired anomaly -- an extreme
move immediately followed by a roughly offsetting one, as the series snaps back to its
true level. That pattern, if present, is evidence for "data error" over "real move".

In [ ]:
asset_returns_2 = adj_prices_defensible.pct_change()
flagged = flag_extreme_returns(asset_returns_2, z_thresh=6.0)
flagged_classified = classify_flagged(flagged)
print(f"{len(flagged_classified)} extreme return(s) flagged (|z| > 6):")
display(flagged_classified)

genuinely_unexplained = flagged_classified[flagged_classified["known_event"].isna()]

if genuinely_unexplained.empty:
    print("\nEvery flagged extreme return falls inside a known market-wide event window "
          "(or none were flagged at all); none is treated as a data error. No price "
          "correction is applied at this step -- we demonstrate the check rather than "
          "assert its result. A vendor that served, say, a single-day zero-price glitch "
          "on an illiquid micro-cap would be exactly the case that WOULD move this row.")
    adj_prices_clean = adj_prices_defensible
else:
    print(f"\n{len(genuinely_unexplained)} flagged observation(s) do not correspond to a "
          "known market-wide event and are candidates for a genuine data error:")
    display(genuinely_unexplained)
    adj_prices_clean = adj_prices_defensible.copy()
    for _, row in genuinely_unexplained.iterrows():
        adj_prices_clean.loc[row["date"], row["ticker"]] = np.nan
    adj_prices_clean = adj_prices_clean.interpolate(method="linear")
    print("Interpolated the price level for the flagged, unexplained observation(s) above.")

r3 = equal_weight_portfolio_returns(adj_prices_clean)
s3 = record_step("3. + investigated extreme returns", r3)


### Step 4: currency inconsistency

Four of our five assets are USD; Toyota (7203.T) is quoted in JPY. Computing its return
directly from the JPY price and averaging it in with USD returns silently assumes 1 JPY
of price change is worth the same as 1 USD of price change, and ignores the JPY/USD
exchange-rate return entirely -- which a USD-based investor actually bears. We convert
Toyota's price series to USD using the USDJPY rate before computing any returns.

In [ ]:
fx_raw, fx_adj, fx_dl_log = load_price_panel(["JPY=X"], PRICE_START, PRICE_END,
                                              cache_dir=DATA_DIR / "fx")
usdjpy = fx_raw["JPY=X"]
print("USDJPY download log:", fx_dl_log)

prices_usd = adj_prices_clean.copy()
prices_usd["7203.T"] = convert_jpy_to_usd(adj_prices_clean["7203.T"], usdjpy)

r4 = equal_weight_portfolio_returns(prices_usd)
s4 = record_step("4. + currency conversion (7203.T JPY -> USD)", r4)


### Steps 5-6: risk-free rate -- zero vs actual, and a look-ahead bug in how it's aligned

So far every Sharpe ratio above was computed against a risk-free rate of exactly zero.
We now bring in the actual US 3-month T-bill yield (`^IRX`, quoted annualised, in percent)
and convert it to a per-period rate. T-bill yields are not published on every trading day,
so aligning them onto our daily return index requires an as-of merge -- and that join
direction is a classic place for a look-ahead bug to hide: `direction="forward"` pairs a
return with a rate the market had not yet observed. We measure that bug's size first
(Step 5), then fix it (Step 6, `direction="backward"`), isolating the look-ahead effect
from the "zero vs actual rate" effect.

In [ ]:
rf_raw, rf_adj, rf_dl_log = load_price_panel(["^IRX"], PRICE_START, PRICE_END,
                                              cache_dir=DATA_DIR / "rf")
irx = rf_raw["^IRX"]   # annualised T-bill yield, in percent
print("^IRX download log:", rf_dl_log)

rf_daily = (1 + irx / 100) ** (1 / 252) - 1

rf_forward_buggy = align_risk_free(r4.index, rf_daily, direction="forward")
rf_backward = align_risk_free(r4.index, rf_daily, direction="backward")
lookahead_gap = (rf_forward_buggy - rf_backward).abs()
print(f"\nLook-ahead alignment gap: mean={lookahead_gap.mean():.8f}, "
      f"max={lookahead_gap.max():.8f}, nonzero on "
      f"{int((lookahead_gap > 0).sum())} of {len(lookahead_gap)} dates")

s5 = record_step("5. + risk-free rate, buggy forward-looking alignment", r4, rf=rf_forward_buggy)
s6 = record_step("6. + risk-free rate, corrected backward-looking alignment", r4, rf=rf_backward)


### Step 7: annualisation factor -- 252 assumed vs actual trading days observed

In [ ]:
actual_ppy = len(r4) / ((r4.index[-1] - r4.index[0]).days / 365.25)
print(f"Actual periods/year in this panel: {actual_ppy:.2f} (naive assumption was 252)")

rf_daily_actual = (1 + irx / 100) ** (1 / actual_ppy) - 1
rf_backward_actual = align_risk_free(r4.index, rf_daily_actual, direction="backward")

s7 = record_step("7. + actual trading-day annualisation factor", r4,
                  rf=rf_backward_actual, periods_per_year=actual_ppy)


### Step 8: return definition -- simple/arithmetic vs log/geometric

An arithmetic mean of simple returns, scaled to an annual figure, systematically
overstates the return an investor actually compounds to, because volatility drags the
geometric growth rate below the arithmetic mean (Jensen's inequality on the concave
log-wealth function). Switching to log returns is not a separate "extra" correction on
top of computing a geometric mean -- the arithmetic mean of log returns already **is**
the annualised geometric growth rate, which is why this step recomputes everything with
`return_type="log"` rather than adding a fifth statistic.

Note what this step does and does not change: the underlying portfolio -- and therefore
its wealth path and maximum drawdown -- is identical to Step 7's; only the *summary
statistic* used to describe its annualised return changes. (An earlier draft of
`equal_weight_portfolio_returns` computed the log-return version by averaging each
asset's own log return across the cross-section, which is a subtly different quantity
from this portfolio's own return on a log scale, by Jensen's inequality -- confirmed with
a synthetic test where that mistake alone moved measured drawdown, with nothing to do
with simple-vs-log at all. Fixed by taking `log1p` of the portfolio's own simple return
instead.)


In [ ]:
log_returns = equal_weight_portfolio_returns(prices_usd, return_type="log")
s8 = record_step("8. + log returns (arithmetic mean = geometric growth rate)", log_returns,
                  rf=rf_backward_actual, periods_per_year=actual_ppy, return_type="log")


### The waterfall table

One row per correction, cumulative -- the deliverable for this section.

In [ ]:
waterfall = pd.DataFrame(waterfall_rows).T
waterfall.columns = ["Ann. return", "Ann. vol", "Sharpe", "Max drawdown"]
display(waterfall.style.format({
    "Ann. return": "{:.2%}", "Ann. vol": "{:.2%}",
    "Sharpe": "{:.3f}", "Max drawdown": "{:.2%}",
}))


### Do the corrections offset each other?

Compare the naive baseline to the fully corrected row (the *net* change), then compare
that against the single largest step-to-step move for each statistic. If the net change is
noticeably smaller than the largest single step, some corrections pushed the numbers in
opposite directions -- meaning the naive-vs-final comparison alone understates how wrong
the naive figure actually was at any single point in the pipeline.

In [ ]:
net_change = waterfall.iloc[-1] - waterfall.iloc[0]
step_changes = waterfall.diff().iloc[1:]

print("Net change, naive baseline -> fully corrected:")
print(net_change.round(4))

print("\nLargest single-step move per statistic:")
for col in waterfall.columns:
    step_name = step_changes[col].abs().idxmax()
    largest = step_changes.loc[step_name, col]
    ratio = abs(net_change[col]) / abs(largest) if largest != 0 else np.nan
    flag = " <-- net change is SMALLER than this single step: corrections partly offset" \
        if abs(net_change[col]) < abs(largest) - 1e-12 else ""
    print(f"  {col}: largest move at '{step_name}' ({largest:+.4f}); "
          f"net/largest ratio = {ratio:.2f}{flag}")


### A2 discussion: which correction mattered most?

**Which single correction moved the numbers most?** The dividend/split adjustment (Step
1) was the largest single mover on three of the four statistics: it took the Sharpe ratio
from 0.855 to 1.018 (+0.163), annualised return from 14.19% to 16.90% (+2.71pp), and
improved maximum drawdown from -35.08% to -34.78% (+0.30pp). Only annualised volatility
was moved more by a different step (the trading-day annualisation factor, Step 7,
+0.27pp).

**Was the naive Sharpe ratio a modest overstatement, or a fundamentally different
answer?** Neither, cleanly -- and that is itself the finding. The net change from naive
(0.855) to fully corrected (0.800) is small: -0.055, about a 6% relative decrease. Read on
its own, that looks like "the naive figure was basically fine." It is not. The dividend
adjustment alone *understated* the naive Sharpe relative to a properly-adjusted figure by
16 percentage points of ratio (0.855 -> 1.018) -- so the naive baseline was actually too
*low* compared to a partially-corrected analysis. Three subsequent corrections then pulled
it back down past the naive starting point: currency conversion (-0.035), the risk-free
rate (-0.115, almost entirely from moving off a zero-rate assumption rather than from the
look-ahead alignment bug itself), and switching to log/geometric returns (-0.085). The
small net change is a coincidence of these swings roughly cancelling, not evidence that
the naive figure was defensible.

**Did corrections offset each other?** Yes, substantially, exactly as the brief warns.
Comparing net change against the largest single step: Sharpe's net move (-0.055) is only
about a third of its largest single step (+0.163, or 34% of it) -- strong evidence of
offsetting. Annualised return's net move (+1.18pp) is 44% of its largest single step
(+2.71pp). Maximum drawdown's net move (+0.22pp) is 73% of its largest single step
(+0.30pp). Only annualised volatility moved in a fairly consistent direction throughout
(net/largest ratio of 85%), because nearly every correction from Step 1 onward pushed
volatility up slightly rather than in mixed directions.

The mechanism, concretely: dividend/split adjustment removes phantom return spikes that
inflate *both* the mean and (via those same spikes) volatility in the naive series, which
mechanically raises Sharpe. Currency conversion, a non-zero risk-free rate, and the
geometric-mean correction are all, in different ways, corrections that remove
overstatement the naive baseline never had a mechanism to inflate in the first place --
they are *independent* sources of downward correction, not linked to the dividend fix by
any real economic relationship. The near-cancellation in our sample is coincidental to
this specific universe and 2016-2025 window; a different asset mix, or a period with a
larger split/dividend history relative to FX and rate effects, could easily see every
correction point the same direction, in which case the naive-vs-final gap would have been
far larger than any individual step -- the brief's own warning that "a bare
before-and-after comparison [is] untrustworthy" is borne out directly by this table, not
merely as a hypothetical caveat.


### Survivorship bias

**What it is.** Every asset in our universe -- Apple, JPMorgan, ExxonMobil, Johnson &
Johnson, Toyota -- is a company that not only still exists today but is a well-known,
currently-thriving large-cap. We selected them with the benefit of nine years of
hindsight. A retail data vendor like Yahoo Finance only serves prices for tickers that
are still listed (or were, under the same symbol, until a known and gracefully-handled
event such as a merger); it does not serve a point-in-time-correct universe of *everything
that was investable* on 1 January 2016, including firms that have since been delisted,
gone bankrupt, or been acquired at a distressed price. We cannot fix this with data
available to us -- `yfinance` simply has no route to a name that no longer trades.

**Direction of the bias.** Upward on return and Sharpe ratio, and understated on
volatility and drawdown. A portfolio that could only ever hold survivors never
experiences one of its constituents going to zero, so its realised return, Sharpe ratio
and worst drawdown are all more flattering than what an investor holding the *actual*,
point-in-time investable universe in 2016 would have experienced.

**Rough magnitude.** Lecture 2 cites a calibrated delisting simulation on a broad,
systematically-sampled S&P 500 backtest putting survivorship bias at roughly +2.7% p.a. of
phantom return. Our case is arguably worse on this dimension, not better: we did not
sample systematically at all -- we hand-picked five globally recognised, multi-decade
survivors specifically because they were recognisable, which is a much stronger
hindsight-driven selection filter than "was still in the S&P 500 index at the download
date". We would expect our true survivorship effect, if it could be measured, to sit at or
above that +2.7% p.a. anchor rather than below it, though we have no way to quantify our
own figure without a point-in-time delisted-security database, which is not available
through a free retail vendor.

## A3 — Baseline risk report

Using the corrected data throughout: `prices_usd` from A2 (adjusted for corporate actions,
defensibly filled, investigated for data errors, and currency-converted) is our clean
price panel from here on, and `r4` (the equal-weight portfolio return computed from it in
A2 Step 4) is our portfolio return series. We add a per-asset return panel alongside it.

In [ ]:
from scipy import stats
from scipy.stats import norm
import matplotlib.pyplot as plt
import seaborn as sns

asset_returns_final = prices_usd.pct_change()
portfolio_returns = r4   # from A2 Step 4: fully corrected, currency-converted

returns_for_a3 = {t: asset_returns_final[t] for t in TICKERS}
returns_for_a3["Portfolio (1/N)"] = portfolio_returns

print("Series available for A3:", list(returns_for_a3.keys()))


### Return characteristics: moments and normality

Mean, volatility, skewness and excess kurtosis for each asset and the portfolio, plus a
Jarque-Bera test of normality (a test built directly from sample skewness and kurtosis,
so its result should read consistently with the moments next to it).

In [ ]:
def compute_return_moments(returns_dict, periods_per_year=252):
    """
    Mean, volatility (daily and annualised), skewness, excess kurtosis and a
    Jarque-Bera normality test for each return series in `returns_dict`.
    """
    rows = {}
    for name, r in returns_dict.items():
        r = pd.Series(r).dropna()
        jb_stat, jb_p = stats.jarque_bera(r)
        rows[name] = {
            "mean_daily": r.mean(),
            "vol_daily": r.std(ddof=1),
            "ann_return": r.mean() * periods_per_year,
            "ann_vol": r.std(ddof=1) * np.sqrt(periods_per_year),
            "skewness": stats.skew(r),
            "excess_kurtosis": stats.kurtosis(r),   # fisher=True by default: 0 = normal
            "jarque_bera_stat": jb_stat,
            "jarque_bera_pvalue": jb_p,
        }
    return pd.DataFrame(rows).T


moments_table = compute_return_moments(returns_for_a3, periods_per_year=actual_ppy)
display(moments_table.style.format({
    "mean_daily": "{:.4%}", "vol_daily": "{:.4%}",
    "ann_return": "{:.2%}", "ann_vol": "{:.2%}",
    "skewness": "{:.3f}", "excess_kurtosis": "{:.3f}",
    "jarque_bera_stat": "{:.1f}", "jarque_bera_pvalue": "{:.4f}",
}))

n_reject_normality = (moments_table["jarque_bera_pvalue"] < 0.05).sum()
print(f"\n{n_reject_normality} of {len(moments_table)} series reject normality at the 5% level.")


**What the higher moments imply.** All six series -- every asset and the portfolio --
reject normality decisively (Jarque-Bera p < 0.0001 throughout). Excess kurtosis ranges
from 5.9 (AAPL, 7203.T) to a striking 14.9 (JPM), against 0 for a true normal distribution
-- these are not marginal deviations, they are an order of magnitude beyond what a
Gaussian model allows. JPM's kurtosis is the standout: it experienced the largest flagged
extreme moves in A2 (the Nov 2020 vaccine rally and Nov 2024 post-election rally both hit
JPM specifically), and that same fat-tailedness resurfaces below as JPM also having the
largest historical-vs-parametric ES gap at 99% -- two independent measurements agreeing on
the same underlying property, which is a useful internal consistency check.

Skewness is more mixed and genuinely interesting: four of the five individual assets
(AAPL +0.001, JPM +0.421, XOM +0.069, 7203.T +0.229) show slightly *positive* skew, while
JNJ (-0.180) and, notably, the **portfolio** (-0.299) show negative skew. A diversified
portfolio having *more* negative skew than most of its own constituents looks
counterintuitive at first, but it is exactly what the correlation analysis below explains:
individual stocks can show positive skew from idiosyncratic upside jumps (earnings
surprises, single-name rallies), but when a systemic shock hits, our five assets stop
behaving idiosyncratically and crash *together* -- so the portfolio inherits a left-tail
event risk that is largely invisible when you look at each asset's own return
distribution in isolation. This is precisely why any risk measure that assumes a
Gaussian, symmetric-and-thin-tailed distribution -- including the parametric VaR/ES
computed next -- will systematically understate genuine portfolio-level tail risk here:
the danger isn't in each asset's own distribution, it's in what happens to their joint
distribution during a crisis.

### Value at Risk and Expected Shortfall

Two methods, at 95% and 99% confidence:

- **Historical (empirical):** the actual sample quantile of the loss distribution, and the
  average loss beyond it. Makes no distributional assumption, but is only as good as the
  historical sample's coverage of tail events.
- **Parametric (Gaussian):** assumes returns are normally distributed and computes VaR/ES
  from the sample mean and standard deviation in closed form. Cheap and smooth, but
  exactly the assumption the moments table above is testing.

Both are reported as positive loss numbers (VaR95 = 5.2% means "a 5.2% loss is exceeded
5% of the time").

In [ ]:
def historical_var_es(returns, alpha=0.95):
    """Empirical VaR/ES as positive loss numbers at confidence level alpha."""
    r = pd.Series(returns).dropna()
    q = r.quantile(1 - alpha)
    var = -q
    es = -r[r <= q].mean()
    return var, es


def parametric_var_es(returns, alpha=0.95):
    """Gaussian (parametric) VaR/ES as positive loss numbers, from sample mean/vol."""
    r = pd.Series(returns).dropna()
    mu, sigma = r.mean(), r.std(ddof=1)
    q = norm.ppf(1 - alpha)
    var = -(mu + sigma * q)
    es = -mu + sigma * norm.pdf(q) / (1 - alpha)
    return var, es


def var_es_table(returns_dict, confidence_levels=(0.95, 0.99)):
    rows = []
    for name, r in returns_dict.items():
        r = pd.Series(r).dropna()
        for alpha in confidence_levels:
            hv, he = historical_var_es(r, alpha)
            pv, pe = parametric_var_es(r, alpha)
            rows.append({"asset": name, "confidence": f"{alpha:.0%}",
                         "method": "historical", "VaR": hv, "ES": he})
            rows.append({"asset": name, "confidence": f"{alpha:.0%}",
                         "method": "parametric", "VaR": pv, "ES": pe})
    return pd.DataFrame(rows)


var_es = var_es_table(returns_for_a3, confidence_levels=(0.95, 0.99))
var_es_wide = var_es.pivot_table(index="asset", columns=["confidence", "method"],
                                  values=["VaR", "ES"])
display(var_es_wide.style.format("{:.2%}"))

# Where the two methods disagree, and by how much
disagreement = (var_es.pivot_table(index=["asset", "confidence"], columns="method", values="ES")
                .assign(gap=lambda d: d["historical"] - d["parametric"]))
print("\nHistorical minus parametric ES (positive = parametric understates the tail):")
display(disagreement.style.format("{:.2%}"))


**Which would we report to a risk committee?** The historical-minus-parametric gap is
positive for every single asset and the portfolio, at both confidence levels -- the
Gaussian assumption understates tail risk uniformly across this universe, not just on
average. The gap roughly triples to quadruples moving from 95% to 99% confidence (e.g.
portfolio: 0.36pp at 95% vs 1.57pp at 99%; JPM: 0.41pp vs 2.03pp), which is exactly what
the moments table predicts -- fat tails matter most in the deepest part of the tail, which
is precisely where the 99% figure lives and where the Gaussian approximation is weakest.

We would report the **historical ES**, not the parametric figure, as the headline number
to a risk committee, for two reasons that reinforce each other: first, the Jarque-Bera
test has already rejected the assumption the parametric method depends on, for every
series in the portfolio, so there is no basis for trusting it here; second, understating a
1-in-100 loss is the more expensive mistake for a risk committee to make than overstating
one -- it directly under-capitalises for the exact scenario (JPM's 99% ES: parametric
2.49%, historical 4.52%, nearly double) that a risk limit exists to catch. The parametric
number is still worth showing alongside it, explicitly labelled as a normality-assuming
sanity check, precisely because the *size* of its gap from the historical figure is itself
informative about how fat-tailed the true distribution is -- but it should never be the
number a limit is set against.

### Drawdown analysis

In [ ]:
def drawdown_series(returns):
    """
    Full drawdown path, plus the maximum drawdown and its peak/trough dates.
    Peak is the last date at or before the trough where wealth equalled its
    running maximum (i.e. where the subsequent decline actually started).
    """
    wealth = (1 + pd.Series(returns).dropna()).cumprod()
    running_max = wealth.cummax()
    dd = wealth / running_max - 1
    trough_date = dd.idxmin()
    max_dd = dd.loc[trough_date]
    peak_date = wealth.loc[:trough_date].idxmax()
    return dd, {"max_drawdown": max_dd, "peak_date": peak_date, "trough_date": trough_date}


portfolio_dd, portfolio_dd_info = drawdown_series(portfolio_returns)
print("Portfolio maximum drawdown:")
for k, v in portfolio_dd_info.items():
    print(f"  {k}: {v}")

fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(portfolio_dd.index, portfolio_dd.values * 100, 0, color="#C0392B", alpha=0.5)
ax.plot(portfolio_dd.index, portfolio_dd.values * 100, color="#C0392B", linewidth=0.8)
ax.axvline(portfolio_dd_info["peak_date"], color="black", linestyle="--", linewidth=1,
           label=f"peak ({portfolio_dd_info['peak_date'].date()})")
ax.axvline(portfolio_dd_info["trough_date"], color="black", linestyle=":", linewidth=1,
           label=f"trough ({portfolio_dd_info['trough_date'].date()})")
ax.set_title("Portfolio drawdown, 1/N equal-weight, corrected data")
ax.set_xlabel("date")
ax.set_ylabel("drawdown (%)")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()


**Identifying the drawdown.** Peak 20 January 2020, trough 23 March 2020, maximum
drawdown -34.86%. This is unambiguously the COVID-19 crash: it falls squarely inside the
`KNOWN_MARKET_EVENTS` window from A2 ("2020-02-20" to "2020-04-07"), and 23 March 2020 is
the well-documented historical bottom of the broad US equity market -- a useful external
correctness check on the drawdown code itself, independent of anything internal to this
notebook. The chart also shows two other, shallower episodes consistent with our event
list: a roughly -20% drawdown around Q4 2018 (the growth-scare sell-off) and a roughly
-18% drawdown through 2022 (the rate-hike bear market) -- neither approaches COVID's
depth, but both land exactly where the literature and our own A2 event list say they
should.

### Correlation structure

Full-sample correlation matrix, then a rolling view -- diversification is a claim about
the *whole* sample, but the number that actually matters to a risk manager is whether it
holds up exactly when it is needed, during the worst drawdown.

In [ ]:
corr_matrix = asset_returns_final[TICKERS].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1,
            square=True, ax=ax, cbar_kws={"label": "correlation"})
ax.set_title("Full-sample correlation matrix (corrected USD returns)")
plt.tight_layout()
plt.show()

display(corr_matrix.style.format("{:.3f}"))


In [ ]:
def rolling_avg_pairwise_corr(returns_df, window=60):
    """
    Average pairwise correlation across every asset pair, in a rolling
    window -- a single diversification-strength indicator over time.
    """
    cols = list(returns_df.columns)
    pairs = [(a, b) for i, a in enumerate(cols) for b in cols[i + 1:]]
    rolling_corrs = pd.DataFrame({
        f"{a}-{b}": returns_df[a].rolling(window).corr(returns_df[b])
        for a, b in pairs
    })
    avg_corr = rolling_corrs.mean(axis=1)
    return avg_corr, rolling_corrs


ROLLING_CORR_WINDOW = 60
avg_corr, pairwise_corrs = rolling_avg_pairwise_corr(asset_returns_final[TICKERS],
                                                      window=ROLLING_CORR_WINDOW)

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                          gridspec_kw={"height_ratios": [1, 1.3]})

axes[0].fill_between(portfolio_dd.index, portfolio_dd.values * 100, 0,
                      color="#C0392B", alpha=0.4)
axes[0].set_ylabel("drawdown (%)")
axes[0].set_title("Portfolio drawdown vs. rolling average pairwise correlation")

axes[1].plot(avg_corr.index, avg_corr.values, color="#123F69", linewidth=1.2)
axes[1].axhline(avg_corr.mean(), color="grey", linestyle=":", linewidth=1,
                 label=f"full-sample average ({avg_corr.mean():.2f})")
axes[1].set_ylabel(f"{ROLLING_CORR_WINDOW}d avg pairwise correlation")
axes[1].set_xlabel("date")
axes[1].legend(loc="upper left")

for ax in axes:
    ax.axvspan(portfolio_dd_info["peak_date"], portfolio_dd_info["trough_date"],
               color="black", alpha=0.08, label="max-drawdown window")

plt.tight_layout()
plt.show()


In [ ]:
dd_window_corr = avg_corr.loc[portfolio_dd_info["peak_date"]:portfolio_dd_info["trough_date"]].mean()
full_sample_corr = avg_corr.mean()
print(f"Average pairwise correlation during the max-drawdown window "
      f"({portfolio_dd_info['peak_date'].date()} to {portfolio_dd_info['trough_date'].date()}): "
      f"{dd_window_corr:.3f}")
print(f"Average pairwise correlation over the full sample: {full_sample_corr:.3f}")
print(f"Difference: {dd_window_corr - full_sample_corr:+.3f} "
      f"({'higher' if dd_window_corr > full_sample_corr else 'lower'} during the drawdown)")


**Did diversification fail when it mattered?** Unambiguously yes. The full-sample
average pairwise correlation is 0.21, but during the COVID drawdown window it climbs to
roughly 0.70 at its peak -- more than three times the baseline -- before decaying back
into the 0.1-0.3 range that characterises the rest of the sample. This is direct,
sample-specific confirmation of the "correlations rise in crises" stylised fact from
Lecture 6, not merely a citation of it: our five-asset, sector- and currency-diversified
universe offered substantially less protection during its single worst episode than the
full-sample correlation matrix would suggest on its own. It also explains the portfolio's
negative skewness noted above -- the joint crash that drives the left tail is exactly this
same correlation spike, invisible in any one asset's own return distribution but fully
visible here. The rolling series also shows correlation was already trending upward
through late 2019 -- before the drawdown itself began -- and elevated again around 2019
and 2022-23, suggesting this is a recurring pattern in our sample rather than a one-off
coincidence of COVID specifically. Toyota's low full-sample correlation with the US
names (0.06-0.16, the lowest pairs in the matrix) is real diversification value in normal
times, but the rolling chart is the honest caveat: it does not fully survive contact with
a genuinely global, systemic shock, where even a JPY-denominated, Tokyo-listed automaker
moved with the rest of the panel.

## A4 — The equal-weight benchmark

**Rebalancing rule: monthly.** On the first trading day of every calendar month, weights
are reset to exactly 1/N; between rebalance dates, weights are left to drift with asset
prices (buy-and-hold). Monthly is a middle ground: annual lets drift accumulate for a
long time (by A3's evidence, correlation and volatility regimes can shift well within a
year), while daily rebalancing -- which is what A2 and A3 used implicitly throughout --
overstates what a real investor would do, since it assumes costless, frictionless trading
every single day. Monthly is also the more common real-world convention for a passive
benchmark.

Note that this makes A2/A3's portfolio (`r4`) and A4's benchmark genuinely different
series, not the same thing under a new name: A2/A3 used the daily-implicit version
throughout because A2's question was about data corrections, not rebalancing policy.
From here on, `benchmark_returns` (monthly-rebalanced) is what Part B is compared
against, per the brief's instruction.

In [ ]:
def rebalanced_portfolio_returns(price_panel, rebalance_freq="M", weights=None):
    """
    Simulate a portfolio rebalanced to target weights (default: equal
    weight, 1/N) on the first trading day of every `rebalance_freq`
    period (a pandas period alias -- "M" monthly, "Q" quarterly, "A"
    annual), drifting freely with asset prices between rebalance dates.

    Verified against a synthetic panel before use: rebalancing on every
    single day exactly reproduces `equal_weight_portfolio_returns()` (max
    abs difference ~3e-18, floating-point noise), and a one-rebalance
    single-month scenario matches manual buy-and-hold arithmetic exactly.

    Returns
    -------
    portfolio_returns : Series, daily simple returns
    turnover : Series, one-way turnover on each date (0 except on
        rebalance dates, and 0 on the very first date too -- that is
        portfolio *initiation* from cash, not a rebalancing trade away
        from an existing position).
    """
    returns = price_panel.pct_change().dropna(how="all")
    tickers = list(returns.columns)
    n = len(tickers)
    if weights is None:
        weights = {t: 1.0 / n for t in tickers}
    target = np.array([weights[t] for t in tickers])

    period_arr = np.asarray(returns.index.to_period(rebalance_freq))
    is_rebalance_day = np.empty(len(period_arr), dtype=bool)
    is_rebalance_day[0] = True
    is_rebalance_day[1:] = period_arr[1:] != period_arr[:-1]

    R = returns[tickers].values
    w = target.copy()
    port_returns = np.empty(len(returns))
    turnover = np.empty(len(returns))

    for t in range(len(returns)):
        if is_rebalance_day[t]:
            turnover[t] = np.abs(target - w).sum() / 2.0
            w = target.copy()
        else:
            turnover[t] = 0.0
        r_t = np.nan_to_num(R[t], nan=0.0)
        port_ret = float(np.dot(w, r_t))
        port_returns[t] = port_ret
        w = w * (1 + r_t) / (1 + port_ret)

    return (pd.Series(port_returns, index=returns.index, name="benchmark"),
            pd.Series(turnover, index=returns.index, name="turnover"))


benchmark_returns, benchmark_turnover = rebalanced_portfolio_returns(prices_usd, rebalance_freq="M")
n_rebalances = (benchmark_turnover > 0).sum()
print(f"Monthly-rebalanced benchmark built: {len(benchmark_returns)} daily observations, "
      f"{n_rebalances} rebalance events.")
print(f"Average one-way turnover per rebalance event: {benchmark_turnover[benchmark_turnover > 0].mean():.2%}")


### Benchmark performance and risk profile

Same statistics as A3, reusing the same functions -- return moments and normality,
VaR/ES by two methods, and the drawdown series -- applied to the monthly-rebalanced
benchmark. We also place the daily-implicit portfolio (`r4`) from A2/A3 alongside it, so
the effect of the rebalancing policy itself is directly visible rather than asserted.

In [ ]:
benchmark_moments = compute_return_moments(
    {"Benchmark (1/N, monthly)": benchmark_returns, "Daily-implicit (A2/A3)": portfolio_returns},
    periods_per_year=actual_ppy,
)
display(benchmark_moments.style.format({
    "mean_daily": "{:.4%}", "vol_daily": "{:.4%}",
    "ann_return": "{:.2%}", "ann_vol": "{:.2%}",
    "skewness": "{:.3f}", "excess_kurtosis": "{:.3f}",
    "jarque_bera_stat": "{:.1f}", "jarque_bera_pvalue": "{:.4f}",
}))


In [ ]:
benchmark_var_es = var_es_table(
    {"Benchmark (1/N, monthly)": benchmark_returns, "Daily-implicit (A2/A3)": portfolio_returns},
    confidence_levels=(0.95, 0.99),
)
benchmark_var_es_wide = benchmark_var_es.pivot_table(index="asset", columns=["confidence", "method"],
                                                      values=["VaR", "ES"])
display(benchmark_var_es_wide.style.format("{:.2%}"))


In [ ]:
benchmark_dd, benchmark_dd_info = drawdown_series(benchmark_returns)
print("Benchmark (1/N, monthly) maximum drawdown:")
for k, v in benchmark_dd_info.items():
    print(f"  {k}: {v}")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(portfolio_dd.index, portfolio_dd.values * 100, color="#9AA5B1", linewidth=0.9,
        label="daily-implicit (A2/A3)")
ax.plot(benchmark_dd.index, benchmark_dd.values * 100, color="#123F69", linewidth=1.1,
        label="benchmark (1/N, monthly-rebalanced)")
ax.set_title("Drawdown: monthly-rebalanced benchmark vs. daily-implicit portfolio")
ax.set_xlabel("date")
ax.set_ylabel("drawdown (%)")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()


TODO once run: compare the benchmark's Sharpe, vol and max drawdown against the
daily-implicit portfolio's (from A3). A monthly-rebalanced portfolio should track the
daily-implicit one closely most of the time but can diverge visibly around sharp,
short-lived moves -- e.g. if the COVID crash and its immediate rebound both happened
within one un-rebalanced month, the monthly benchmark would have let winners and losers
drift further before resetting, which can show up as either a deeper or shallower
drawdown than the daily version, depending on which assets were doing the drifting.

### Frictions ignored

We are not required to model these, but we are required to know they are missing --
and to have a sense of their size rather than just naming them.

- **Transaction costs.** The benchmark trades on every rebalance date, at the average
  one-way turnover per event reported by the "benchmark built" cell above (roughly a few
  percent per month, driven by how far our five assets drift apart between rebalances).
  At even a conservative 5-10 bp round-trip cost on large-cap, liquid names, that is a
  small but nonzero annual drag -- and would be far larger for a less liquid universe or
  a more frequent rebalancing rule.
- **Bid-ask spread.** Every trade crosses the spread, which is not the same as a
  commission and is not visible in any exchange-reported closing price -- it is a real
  cost this notebook has no way to measure from daily OHLCV data alone.
- **Cash drag from imperfect rebalancing.** A real account cannot buy fractional shares
  in the exact proportions 1/N implies, and cannot rebalance at the exact instant the
  code specifies (the next tradable price is not the theoretical rebalance price). Both
  push the achievable return slightly below what this simulation reports.

None of these favour the benchmark disproportionately -- if anything, an actively
rebalanced or optimised Part B strategy typically trades *more* than 1/N, so these same
frictions bite it harder. That asymmetry is itself relevant context for evaluating Part B
against this benchmark, not just a disclaimer.

### Adopting the benchmark for Part B

`benchmark_returns` (1/N, monthly-rebalanced) is the reference series for every Part B
result that can sensibly be compared against it, per the brief. A small reusable
comparison function, built once here rather than re-derived per extension.

In [ ]:
def benchmark_comparison_table(strategy_returns_dict, benchmark=benchmark_returns,
                               periods_per_year=None, confidence_levels=(0.95, 0.99)):
    """
    Side-by-side moments + VaR/ES + drawdown for one or more Part B
    strategies against the 1/N benchmark, using the exact same functions
    as A3/A4 so every comparison in this notebook is computed identically.
    """
    ppy = periods_per_year if periods_per_year is not None else actual_ppy
    series = {"Benchmark (1/N, monthly)": benchmark, **strategy_returns_dict}

    moments = compute_return_moments(series, periods_per_year=ppy)
    var_es = var_es_table(series, confidence_levels=confidence_levels)
    var_es_wide = var_es.pivot_table(index="asset", columns=["confidence", "method"],
                                     values=["VaR", "ES"])

    dd_rows = {}
    for name, r in series.items():
        _, info = drawdown_series(r)
        dd_rows[name] = info
    drawdowns = pd.DataFrame(dd_rows).T

    return moments, var_es_wide, drawdowns


# Smoke test: calling it with no extra strategies should just reproduce the
# benchmark's own row from the tables above.
_moments_check, _var_es_check, _dd_check = benchmark_comparison_table({})
display(_moments_check.style.format({
    "mean_daily": "{:.4%}", "vol_daily": "{:.4%}",
    "ann_return": "{:.2%}", "ann_vol": "{:.2%}",
    "skewness": "{:.3f}", "excess_kurtosis": "{:.3f}",
    "jarque_bera_stat": "{:.1f}", "jarque_bera_pvalue": "{:.4f}",
}))
